In [10]:
import shutil
from pathlib import Path

def copy_preserving_full_structure(search_dir, base_prefix, target_prefix, pattern="**/seed_*/final_metrics.json"):
    search_path = Path(search_dir)
    base_path = Path(base_prefix)
    target_path = Path(target_prefix)
    
    # 查找特定文件夹下的目标文件
    files_to_copy = list(search_path.rglob(pattern))
    
    if not files_to_copy:
        print(f"在 {search_dir} 中没有找到匹配的文件。")
        return

    print(f"共找到 {len(files_to_copy)} 个文件，开始复制...\n")
    
    success_count = 0
    for file_path in files_to_copy:
        try:
            # 关键点：不再相对于 search_dir 计算，而是相对于你的根目录(results)计算！
            # 这样就能保留中间的 criteo/train_y/TARNET/... 等所有层级
            relative_to_base = file_path.relative_to(base_path)
            
        except ValueError:
            print(f"⚠️ 跳过：文件 {file_path} 不在基础路径 {base_path} 下。")
            continue
            
        # 拼接到最终的输出目录
        final_dest = target_path / relative_to_base
        
        # 确保目标文件的多层父目录都被创建
        final_dest.parent.mkdir(parents=True, exist_ok=True)
        
        # 复制文件
        shutil.copy2(file_path, final_dest)
        success_count += 1
        
        print(f"✅ 成功映射:")
        print(f"   源: {file_path}")
        print(f"   目标: {final_dest}\n")
        
    print(f"🎉 复制完成！成功复制 {success_count} 个文件。")

# ==========================================
# 在这里配置你的路径
# ==========================================

# 1. 你当前想要执行搜索的特定深层目录（你可以随便换其他的上级路径）
my_search_dir = "/NAS/shith/uplift/results_0717/criteo/train_y/TARNET_old/y_v1_base/run_v1_base"

# 2. 原始的总根目录（用来计算相对路径，保留中间的那一坨）
my_base_prefix = "/NAS/shith/uplift/results_0717"

# 3. 想要输出的新总根目录
my_target_prefix = "/NAS/shith/uplift/results_final"

# 执行
copy_preserving_full_structure(my_search_dir, my_base_prefix, my_target_prefix)

共找到 5 个文件，开始复制...

✅ 成功映射:
   源: /NAS/shith/uplift/results_0717/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_1042/final_metrics.json
   目标: /NAS/shith/uplift/results_final/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_1042/final_metrics.json

✅ 成功映射:
   源: /NAS/shith/uplift/results_0717/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_3042/final_metrics.json
   目标: /NAS/shith/uplift/results_final/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_3042/final_metrics.json

✅ 成功映射:
   源: /NAS/shith/uplift/results_0717/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_2042/final_metrics.json
   目标: /NAS/shith/uplift/results_final/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_2042/final_metrics.json

✅ 成功映射:
   源: /NAS/shith/uplift/results_0717/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_4042/final_metrics.json
   目标: /NAS/shith/uplift/results_final/criteo/train_y/TARNET_old/y_v1_base/run_v1_base/seed_4042/final_metrics.json

✅ 成功映射:
   源: /NAS/shith/

In [6]:
import json
from pathlib import Path

def print_sorted_metrics(base_dir, target_metric="Test_Target_Y_AUUC"):
    base_path = Path(base_dir)
    
    # 根据你提供的图片 image_5e51b8.png，这里列出所有的模型文件夹名称
    models = [
        "CFRNet", "DESCN", "DragonNet", "ECUP", "EFIN", 
        "EUEN_Academic", "MOTTO", "MTMT", "S_Learner", "T_Learner"
    ]
    
    # 字典用于存储每个模型的结果: { 'ModelName': [(auuc_value, file_path), ...] }
    results = {model: [] for model in models}
    
    print(f"正在扫描目录: {base_path} ...\n")
    
    # 遍历每个模型文件夹
    for model in models:
        model_dir = base_path / model
        
        if not model_dir.exists():
            continue
            
        # 递归查找该模型下所有的 final_metrics.json
        for json_path in model_dir.rglob("final_metrics.json"):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                # 检查指定的 key 是否在 json 中
                if target_metric in data:
                    metric_val = data[target_metric]
                    # 将数值和绝对路径存入列表
                    results[model].append((metric_val, str(json_path)))
            except Exception as e:
                print(f"⚠️ 读取文件出错跳过 {json_path}: {e}")
                
    # ==========================
    # 打印结果：按模型分组，并按分数降序排序
    # ==========================
    for model in models:
        model_results = results[model]
        
        if not model_results:
            print(f"[{model}]")
            print("  -> 未找到数据或文件夹不存在。\n")
            continue
            
        # 按 AUUC 值 (x[0]) 从高到低排序
        sorted_results = sorted(model_results, key=lambda x: x[0], reverse=True)
        
        print(f"[{model}] - 共找到 {len(sorted_results)} 个结果")
        for val, path in sorted_results:
            # 格式化输出，保留 6 位小数，并输出对应的路径
            print(f"  {target_metric}: {val:.6f} | 路径: {path}")
        print("-" * 80 + "\n")

# ==========================================
# 配置你的基础路径
# ==========================================
# 假设这是包含所有这些模型文件夹的上级目录
my_base_dir = "/NAS/shith/uplift/results_0717/criteo/train_y"

# 执行提取和打印（如果你想看 C 目标的 AUUC，把参数改成 "Test_Target_C_AUUC" 即可）
print_sorted_metrics(my_base_dir, target_metric="Test_Target_Y_AUUC")

正在扫描目录: /NAS/shith/uplift/results_0717/criteo/train_y ...

[CFRNet] - 共找到 8 个结果
  Test_Target_Y_AUUC: 0.900737 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2/final_metrics.json
  Test_Target_Y_AUUC: 0.882171 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2_0714/final_metrics.json
  Test_Target_Y_AUUC: 0.866721 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.847621 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/final_metrics.json
  Test_Target_Y_AUUC: 0.772776 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.284294 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.208651 | 路径: /NAS/shith/u

In [11]:
import shutil
from pathlib import Path

# 将你跑出来的结果作为字符串直接喂给程序
log_text = """
[CFRNet] - 共找到 8 个结果
  Test_Target_Y_AUUC: 0.900737 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2/final_metrics.json
  Test_Target_Y_AUUC: 0.882171 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2_0714/final_metrics.json
  Test_Target_Y_AUUC: 0.866721 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.847621 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/final_metrics.json
  Test_Target_Y_AUUC: 0.772776 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.284294 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.208651 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_2042/final_metrics.json
  Test_Target_Y_AUUC: 0.112761 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_5042/final_metrics.json

[DESCN] - 共找到 1 个结果
  Test_Target_Y_AUUC: 0.873384 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DESCN/y_baseline_descn/run_descn/final_metrics.json

[DragonNet] - 共找到 7 个结果
  Test_Target_Y_AUUC: 0.891591 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/final_metrics.json
  Test_Target_Y_AUUC: 0.888233 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.887240 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/seed_2042/final_metrics.json
  Test_Target_Y_AUUC: 0.884743 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet_0714/final_metrics.json
  Test_Target_Y_AUUC: 0.881889 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/seed_5042/final_metrics.json
  Test_Target_Y_AUUC: 0.881866 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.692637 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/DragonNet/y_baseline_dragonnet/run_dragonnet/seed_4042/final_metrics.json

[ECUP] - 共找到 6 个结果
  Test_Target_Y_AUUC: 0.893687 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/seed_5042/final_metrics.json
  Test_Target_Y_AUUC: 0.893657 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/final_metrics.json
  Test_Target_Y_AUUC: 0.891931 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.884382 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.865146 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.847332 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/ECUP/y_ecup/run_1_65536/seed_2042/final_metrics.json

[EFIN] - 共找到 9 个结果
  Test_Target_Y_AUUC: 0.916860 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/final_metrics.json
  Test_Target_Y_AUUC: 0.909348 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/seed_2042/final_metrics.json
  Test_Target_Y_AUUC: 0.909192 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.906985 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.893633 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin_128/run_efin_rank128/final_metrics.json
  Test_Target_Y_AUUC: 0.620185 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin_32/run_efin_rank32/final_metrics.json
  Test_Target_Y_AUUC: 0.556991 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.536059 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin_64/run_efin_rank64/final_metrics.json
  Test_Target_Y_AUUC: 0.206277 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EFIN/y_baseline_efin/run_efin/seed_5042/final_metrics.json

[EUEN_Academic] - 共找到 1 个结果
  Test_Target_Y_AUUC: 0.872697 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/EUEN_Academic/y_baseline_euen_academic_ms/run_euen_academic/final_metrics.json

[MOTTO] - 共找到 6 个结果
  Test_Target_Y_AUUC: 0.902756 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.899840 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.897453 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/seed_5042/final_metrics.json
  Test_Target_Y_AUUC: 0.894252 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.890921 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/final_metrics.json
  Test_Target_Y_AUUC: 0.880841 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MOTTO/y_motto/motto_kdd25_official_20260409/seed_2042/final_metrics.json

[MTMT] - 共找到 6 个结果
  Test_Target_Y_AUUC: 0.881174 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/final_metrics.json
  Test_Target_Y_AUUC: 0.859015 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.852676 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/seed_5042/final_metrics.json
  Test_Target_Y_AUUC: 0.788466 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/seed_2042/final_metrics.json
  Test_Target_Y_AUUC: 0.694408 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.535514 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/MTMT/y_mtmt_mlp/mtmt_mlp_nas_20260409/seed_4042/final_metrics.json

[S_Learner] - 共找到 6 个结果
  Test_Target_Y_AUUC: 0.902565 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.901682 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/final_metrics.json
  Test_Target_Y_AUUC: 0.881104 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/seed_3042/final_metrics.json
  Test_Target_Y_AUUC: 0.872058 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.732519 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/5042/final_metrics.json
  Test_Target_Y_AUUC: 0.663624 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/S_Learner/y_baseline_s_learner/run_s_learner/seed_2042/final_metrics.json

[T_Learner] - 共找到 6 个结果
  Test_Target_Y_AUUC: 0.896190 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/final_metrics.json
  Test_Target_Y_AUUC: 0.880619 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/seed_2042/final_metrics.json
  Test_Target_Y_AUUC: 0.877912 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/seed_5042/final_metrics.json
  Test_Target_Y_AUUC: 0.871957 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/seed_1042/final_metrics.json
  Test_Target_Y_AUUC: 0.869649 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/seed_4042/final_metrics.json
  Test_Target_Y_AUUC: 0.869586 | 路径: /NAS/shith/uplift/results_0717/criteo/train_y/T_Learner/y_baseline_t_learner/run_t_learner/seed_3042/final_metrics.json
"""

# 解析文本结构
model_paths = {}
current_model = None

for line in log_text.strip().split('\n'):
    line = line.strip()
    if line.startswith('[') and ']' in line and '共找到' in line:
        current_model = line.split(']')[0].strip('[')
        model_paths[current_model] = []
    elif '路径:' in line and current_model:
        path = line.split('路径:')[1].strip()
        model_paths[current_model].append(path)

# 定义输出根目录和目标 seed 名称
target_base_dir = Path("/NAS/shith/uplift/results_final")
target_seeds = ["seed_42", "seed_1042", "seed_2042"]

print(f"==================== 开始提取 JSON 到 results_final ====================")

for model, paths in model_paths.items():
    if not paths:
        continue
        
    # EFIN 取 2-4（索引 1, 2, 3），其他取 1-3（索引 0, 1, 2）
    if model == "EFIN":
        selected_paths = paths[1:4]
    else:
        selected_paths = paths[0:3]
        
    print(f"\n处理模型: {model}")
    
    for i, path_str in enumerate(selected_paths):
        src_json_file = Path(path_str)
        
        # 目标文件夹的名称
        new_seed_name = target_seeds[i]
        
        # 目标 JSON 文件的绝对路径: /NAS/.../results_final/模型名/seed_xxx/final_metrics.json
        target_dir = target_base_dir / model / new_seed_name
        target_json_file = target_dir / "final_metrics.json"
        
        print(f"  [{i+1}] 正在复制:")
        print(f"      源文件: {src_json_file}")
        print(f"      至目标: {target_json_file}")
        
        try:
            # 必须先确保目标目录被创建，否则 copy2 会报错
            target_dir.mkdir(parents=True, exist_ok=True)
            
            # 使用 shutil.copy2 单独复制 json 文件，保留元数据
            shutil.copy2(src_json_file, target_json_file)
            print("      ✅ 复制成功")
        except Exception as e:
            print(f"      ❌ 复制失败: {e}")

print("\n==================== JSON 提取完毕！ ====================")

==================== 开始提取 JSON 到 results_final ====================

处理模型: CFRNet
  [1] 正在复制:
      源文件: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2/final_metrics.json
      至目标: /NAS/shith/uplift/results_final/CFRNet/seed_42/final_metrics.json
      ✅ 复制成功
  [2] 正在复制:
      源文件: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet_2/run_cfrnet_2_0714/final_metrics.json
      至目标: /NAS/shith/uplift/results_final/CFRNet/seed_1042/final_metrics.json
      ✅ 复制成功
  [3] 正在复制:
      源文件: /NAS/shith/uplift/results_0717/criteo/train_y/CFRNet/y_baseline_cfrnet/run_cfrnet/seed_4042/final_metrics.json
      至目标: /NAS/shith/uplift/results_final/CFRNet/seed_2042/final_metrics.json
      ✅ 复制成功

处理模型: DESCN
  [1] 正在复制:
      源文件: /NAS/shith/uplift/results_0717/criteo/train_y/DESCN/y_baseline_descn/run_descn/final_metrics.json
      至目标: /NAS/shith/uplift/results_final/DESCN/seed_42/final_metrics.json
      ✅ 复制成功

处理模型: DragonNet
  [1] 正在复